# Synthetic asset dataset generator

There are three types of variables to consider. Variables that are completely independent from other observed variables (Missing Completely At Random or MCAR).

The generator writes one measured file and one truth file for every combination of two controls:

- `completion`: the chance that a cell stays visible. Every cell in every column is hidden independently.
- `noise`: the bound of a uniform multiplicative error on the noisy readings. Humidity, salt, overgrowth, the placebo and the three gas readings are noisy because of how they are measured. All other columns, including `life`, `total_gas` and `leakage`, are exact.

All nine scenarios describe the same assets. Each cell keeps one noise draw, scaled by the noise level, and one mask draw, compared with the completion level. A cell hidden at 0.9 completion is therefore also hidden at 0.8 and 0.7, and scenarios differ only through the two controls. Run the notebook again with the same `SEED` to reproduce the files exactly.

In [ ]:
import csv
import random
from pathlib import Path

SEED = 219
COMPLETION_RATIOS = (0.7, 0.8, 0.9)
NOISE_RATIOS = (0.0, 0.05, 0.1)
OUTPUT = Path.cwd().parent if Path.cwd().name == "generator" else Path("data/datasets")

HEADERS = ["x", "y", "install_date", "model_id", "altitude", "measured_humid", "measured_salt",
           "measured_overgrowth", "life", "random_placebo", "measured_gas_a", "measured_gas_b",
           "measured_gas_c", "total_gas", "leakage"]
# Only these columns are noisy, because of how they are measured. The rest are exact by design.
NOISY = {"measured_humid", "measured_salt", "measured_overgrowth", "random_placebo",
         "measured_gas_a", "measured_gas_b", "measured_gas_c"}

rng = random.Random(SEED)


def draw_model(day_count):
    """Newer models replace older ones as installation days pass."""
    if day_count < 200:
        return 0
    if day_count < 400:
        return rng.randint(0, 1)
    if day_count < 700:
        return rng.randint(0, 2)
    if day_count < 1000:
        return rng.randint(1, 3)
    if day_count < 1200:
        return rng.randint(2, 3)
    return 3


def draw_gases(model, day_count, leakage):
    """Gas levels fall with age and leakage; each model starts high in a different gas."""
    hidden_var_a = rng.uniform(0.975, 1.025)
    hidden_var_b = rng.uniform(0.975, 1.025)
    hidden_var_c = rng.uniform(0.975, 1.025)
    start_a, start_b, start_c = {
        0: (10000, 10000, 10000),
        1: (12000, 9000, 9000),
        2: (9000, 12000, 9000),
        3: (9000, 9000, 12000),
    }[model]
    gas_a = max(hidden_var_a * start_a - day_count * leakage, 0)
    gas_b = max(hidden_var_b * start_b - 0.0005 * day_count * day_count * leakage, 0)
    gas_c = max(hidden_var_c * start_c - day_count * leakage - 0.00025 * day_count * day_count * leakage, 0)
    return gas_a, gas_b, gas_c


def remaining_life(model, day_count, humid, salt, overgrowth, total_gas):
    """Each model is exposed to a different mix of hazards; more gas reduces life in all of them."""
    gas_penalty = 25000 * (total_gas / 300000)
    if model == 0:
        return 100000 - day_count * humid - day_count * salt - day_count * overgrowth - gas_penalty
    if model == 1:
        return 100000 - day_count * humid - day_count * overgrowth - gas_penalty
    if model == 2:
        return 100000 - day_count * salt - day_count * overgrowth - gas_penalty
    return 100000 - day_count * overgrowth - gas_penalty


truth = []
day_count = 0
day_num_installs = rng.randint(5, 15)
for i in range(100):
    for j in range(100):
        if day_count % 7 == 0:
            day_count += 2
        model = draw_model(day_count)
        altitude = i + j + 100
        humid = rng.random() * (altitude / 300)
        salt = rng.random() * ((j + 200) / 300)
        overgrowth = max(2 * humid - salt, 0)
        leakage = rng.uniform(0.8, 1.2)
        gas_a, gas_b, gas_c = draw_gases(model, day_count, leakage)
        total_gas = gas_a + gas_b + gas_c
        life = remaining_life(model, day_count, humid, salt, overgrowth, total_gas)
        random_var = rng.random()
        truth.append([i, j, day_count, model, altitude, humid, salt, overgrowth, life, random_var,
                      gas_a, gas_b, gas_c, total_gas, leakage])
        day_num_installs -= 1
        if day_num_installs == 0:
            day_count += 1
            day_num_installs = rng.randint(5, 15)

# One draw per cell, shared by all scenarios.
noise_draws = [[rng.uniform(-1, 1) for _ in HEADERS] for _ in truth]
mask_draws = [[rng.random() for _ in HEADERS] for _ in truth]


def measure(row, noise_row, mask_row, completion_ratio, noise_ratio):
    """Apply measurement error to the noisy columns, then hide cells in every column."""
    result = []
    for name, value, noise, mask in zip(HEADERS, row, noise_row, mask_row):
        measured = value + value * noise * noise_ratio if name in NOISY else value
        result.append(None if mask > completion_ratio else measured)
    return result


def write(path, rows):
    with open(path, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(HEADERS)
        writer.writerows(rows)


for completion_ratio in COMPLETION_RATIOS:
    for noise_ratio in NOISE_RATIOS:
        name = f"completion{completion_ratio}-noise{noise_ratio}"
        measured = [measure(row, noise_row, mask_row, completion_ratio, noise_ratio)
                    for row, noise_row, mask_row in zip(truth, noise_draws, mask_draws)]
        write(OUTPUT / f"{name}-measured.csv", measured)
        write(OUTPUT / f"{name}-truth.csv", truth)
        print("wrote", name)


## Check the generated data

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT / "completion0.9-noise0.0-truth.csv")
df


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(16, 16))
ax = fig.gca()
df.hist(ax=ax)
plt.show()


In [ ]:
corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for row in range(len(corr)):
    for column in range(len(corr)):
        ax.text(column, row, f"{corr.iloc[row, column]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, fraction=0.04)
ax.set_title("Correlation Heatmap")
plt.show()
